In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.functions import (col as c, expr)

pool = "0x8ad599c3a0ff1de082011efddc58f1908eb6e6d8".lower()
start_date = "2021-05-01"
end_date   = "2023-05-01"
bucket = "amm-microstructure-emr-logs"  
prefix = "amm-microstructure"  

In [ ]:
logs = spark.read.parquet("s3://aws-public-blockchain/v1.0/eth/logs/") \
    .where(
        (col("address") == pool) &
        (col("date").between(start_date, end_date))
    )

In [ ]:
logs.write.mode("overwrite").parquet(
    f"s3://{bucket}/{prefix}/univ3_pool_logs.parquet"
)

## Unique transaction hashes in the pool

In [ ]:
pool_txs = logs.select(col("transaction_hash").alias("tx_hash")).distinct()

## Compute ETH fees for those transactions

In [ ]:
txs = spark.read.parquet("s3://aws-public-blockchain/v1.0/eth/transactions/") \
    .where(col("date").between(start_date, end_date))

txs_pool = txs.join(pool_txs, txs.hash == pool_txs.tx_hash, "inner")

txs_pool = txs_pool.withColumn(
    "txn_fee",
    (c("receipt_gas_used") * c("receipt_effective_gas_price")) / expr("1e18")
)

In [ ]:
fees_by_block = txs_pool.groupBy("block_number", "block_timestamp", "date") \
    .sum("txn_fee") \
    .withColumnRenamed("sum(txn_fee)", "pool_related_gas_fee_eth")

In [ ]:
fees_by_block.write.mode("overwrite").parquet(
    f"s3://{bucket}/{prefix}/fees_by_block.parquet"
)

## Block level environment 

In [ ]:
blocks = spark.read.parquet("s3://aws-public-blockchain/v1.0/eth/blocks/") \
    .where(col("date").between(start_date, end_date))


pool_block_numbers = txs_pool.select("block_number").distinct()
env_blocks = blocks.join(pool_block_numbers, "block_number", "inner")

env_blocks.write.mode("overwrite").parquet(
    f"s3://{bucket}/{prefix}/env_blocks.parquet"
)